# Finalizing Sample Data
Applying the final processing methodology on the sample data.

In [1]:
import pandas as pd

## Loading Posts

In [4]:
# load the raw posts
RAW_POSTS_PATH = "data/raw/subreddit_resumes_posts.parquet"
posts = pd.read_parquet(RAW_POSTS_PATH)

## Filtering Data

In [7]:
# ensuring post has good text
invalid_words = ["[removed]", "[deleted]"]
is_text_valid = (posts["selftext"].str.len() > 0) & (posts["selftext"].isin(invalid_words) == False)

In [10]:
# ensuring post has a image url
valid_endings = ["png", "jpeg"]
url_ending = posts["url"].str.split(".").str[-1]
is_url_valid = url_ending.isin(valid_endings) == True

In [11]:
# [X YoE, Current Role/Unemployed, Target Role, Country]
relevant_title_regex = r"\[([^,]+),\s*([^,]+),\s*([^,]+),\s*([^\]]+)\]"
posts[["yoe", "current_role", "target_role", "country"]] = posts["title"].str.extract(relevant_title_regex)

In [18]:
# filtered years of experience
yoe_len = posts["yoe"].str.len()
contains_valid_yoe_phrase = (
    posts["yoe"].str.lower().str.contains("yoe") | 
    posts["yoe"].str.lower().str.contains("years") |
    posts["yoe"].str.lower().str.contains("yrs") |
    posts["yoe"].str.lower().str.contains("yr") |
    posts["yoe"].str.lower().str.contains("year")
)
is_sufficient_yoe_len = (yoe_len > 3.5) & (yoe_len < 7)
is_yoe_valid = is_sufficient_yoe_len & contains_valid_yoe_phrase

In [21]:
# filtered country
is_valid_country = (posts["country"].str.len() > 0) & (posts["country"].str.len() < 26)

In [25]:
# applying all filters
filtered_posts = posts.dropna()[
    is_text_valid
    & is_url_valid
    & is_yoe_valid
    & is_valid_country
]

/var/folders/my/90dxhc_500vbg9nf8jbt4lq80000gn/T/ipykernel_47730/347494935.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  filtered_posts = posts.dropna()[


## Normalizing Data

In [26]:
filtered_posts.shape

(3066, 12)

In [36]:
# parse the years of experience
yoe_parsed = pd.to_numeric(
    filtered_posts["yoe"].str.extract(r"(-?\d+(?:\.\d+)?)", expand=False),
    errors="coerce"
)
filtered_posts["yoe"] = yoe_parsed

In [38]:
filtered_posts["yoe"].describe()

count    3050.000000
mean        2.713213
std         3.612642
min         0.000000
25%         0.000000
50%         2.000000
75%         4.000000
max        45.000000
Name: yoe, dtype: float64

In [52]:
# loading the reference conversion for roles
ref_roles = pd.read_csv("data/intermediate/normalized_roles.csv")
ref_roles = ref_roles[["raw_role", "detailed_code"]]
ref_roles = ref_roles.set_index("raw_role")["detailed_code"]
ref_roles.head(2)

raw_role
Account Executive/Customer Success Hybrid    41-3091
Sub Teacher                                  25-3031
Name: detailed_code, dtype: str

In [60]:
# parse the current roles
parsed_current_roles = filtered_posts["current_role"].map(ref_roles)
parsed_current_roles.notna().sum()
filtered_posts["current_role"] = parsed_current_roles

In [61]:
# parse the target role
parsed_target_roles = filtered_posts["target_role"].map(ref_roles)
parsed_target_roles.notna().sum()
filtered_posts["target_role"] = parsed_target_roles

In [68]:
# loading the reference conversion for countries
ref_country = pd.read_csv("data/intermediate/normalized_countries.csv")
ref_country = ref_country[["raw_country", "cleaned_code"]]
ref_country = ref_country.set_index("raw_country")["cleaned_code"]
ref_country.head(2)

raw_country
Toronto                                             Canada
Looking to Pivot into Customer Service/IT Jobs    UNMAPPED
Name: cleaned_code, dtype: str

In [72]:
# parse the country
parsed_country = filtered_posts["country"].map(ref_country)
filtered_posts["country"] = parsed_country

In [75]:
filtered_posts = filtered_posts.dropna()

In [77]:
filtered_posts.to_parquet("data/final/posts.parquet", index=False)